# Phase 5 — the ruler had a constant in it

Every fit in this project runs at **200 trees, learning rate 0.1**. The probe. All
three GBDT families. Every run, every dataset, since the beginning.

```
probe.py:110      PROBE_PARAMS = {"n_estimators": 200}
models.py:419-432 lgbm / xgb / catboost: n_estimators=200
learning_rate     never set anywhere — whatever the library ships
```

Nothing tunes it. Nothing stops early. 0035 took model parameters away from the agent
deliberately, and 0039 noticed the gap and left it open:

> the terminal phase enumerates library defaults, so it measures the libraries. A fair
> comparison needs a common budget or early stopping — left open.

`HANDOFF.md` promoted this to the leading explanation for why we are below median on
Rossmann, with a specific mechanism: **capacity matters in proportion to rows.**
house-prices has 1,168 rows and reached the top 9%; Rossmann has 916,000 and sits at
rank ~2,200 of 3,303.

This notebook is the test of that. It is also the record of the hypothesis being
**wrong**, of a confound that nearly got written up as the answer, and of the thing
that turned up instead — which is worse than the thing we were looking for.

## 1. Registering the prediction first

0022 is the standing lesson and 0052 is the recent one: a result you did not predict is
a story you told afterwards. So before any fit, in the script's docstring:

* Rossmann and Porto Seguro **gain**; more than 0.02 rmspe on Rossmann.
* house-prices **gains little or loses** — 1,168 rows, more capacity is more variance.
* restaurant, at 227k rows, **lands in between** — it is the out-of-sample test of the
  row-count story, prepared the same day, with no result yet in existence.
* early stopping **roughly matches** the best fixed budget without being told the answer.

Falsifier: **Rossmann gains less than 0.005**, in which case the whole line is closed.

Score one out of four. Keep reading.

> **Running this notebook needs the sweep to have been run first:**
> `python scripts/probe_capacity.py` writes `_runs/capacity_probe.csv`, which every
> table below reads. `_runs/` is gitignored, so on a fresh clone that file will not
> exist. The full sweep is about six minutes, most of it Rossmann's early-stop arm.


In [ ]:
import os, sys
import pandas as pd

REPO = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
sys.path.insert(0, os.path.join(REPO, "src"))
sys.path.insert(0, os.path.join(REPO, "scripts"))

results = pd.read_csv(os.path.join(REPO, "_runs", "capacity_probe.csv"))
print(results.groupby(["dataset", "table"]).size().to_string())

## 2. How the measurement is built

Two properties matter, and both are easy to get wrong.

**The table and the validation rows are held fixed. Only capacity varies.** That is
0017's rule — a comparison differs by exactly one thing.

**The score comes from the harness's own ruler,** `probe_table`, not from a fit written
here. If I reimplemented the fit, every number would be incomparable with the run
journals it is supposed to explain, and any small difference in my reimplementation
would look exactly like a capacity effect.

So the budget is swapped by rebinding the module constant, which is ugly and deliberate:

In [ ]:
import contextlib
from tabular_agent import probe

@contextlib.contextmanager
def swap_probe_params(params):
    original = probe.PROBE_PARAMS
    probe.PROBE_PARAMS = dict(params)
    try:
        yield
    finally:
        probe.PROBE_PARAMS = original

print("the incumbent:", probe.PROBE_PARAMS)
with swap_probe_params({"n_estimators": 5000, "learning_rate": 0.03}):
    print("inside      :", probe.PROBE_PARAMS)
print("restored    :", probe.PROBE_PARAMS)

The early-stopping arm cannot go through that path — it needs an `eval_set` the probe
has no argument for. So it is two steps: fit on the work half minus its last 10% **by
the split column**, so the inner validation is *later* than the inner training and has
the same shape as the real task; read `best_iteration_`; then refit at that many trees
through the ordinary path. The reported number is produced by the same code as every
other row; only the choice of `n_estimators` came from somewhere else.

## 3. The row-count hypothesis, refuted

Four datasets, baseline tables (raw numeric columns, nothing engineered), each on the
split its own runs used.

In [ ]:
def table_for(dataset, table):
    g = results[(results.dataset == dataset) & (results.table == table)]
    return g[["budget", "score", "gain_vs_current"]].to_string(index=False)

for ds in ["house", "restaurant", "seguro", "rossmann"]:
    print(f"--- {ds}")
    print(table_for(ds, "baseline table"))
    print()

Read the two middle rows against the prediction.

| dataset | rows | best fixed budget | early stop |
|---|---|---|---|
| house-prices | 1,168 | −0.0034 (all lose) | 162 → **+0.0062** |
| restaurant | 227,139 | +0.0004 (nothing) | 1,018 → +0.0003 |
| Porto Seguro | 476,170 | −0.0188 (all but one lose) | 181 → **+0.0049** |
| Rossmann | 915,744 | **+0.0573** | 16,926 → **+0.0606** |

Rossmann gains enormously — ten times the registered threshold. And **restaurant, at
227,000 rows, gains nothing at all.** If rows were the mechanism it should sit between
house-prices and Rossmann. It sits with house-prices.

The hypothesis in `HANDOFF.md` is refuted. Ulf's instruction — *check it everywhere, it
is most probably wrong* — was right.

## 4. The confound that was nearly the conclusion

"Restaurant gains nothing, Rossmann gains a lot, therefore it is not rows" was one table
away from being written down. Then: *why* doesn't restaurant gain?

Look at what is actually in the two baseline tables.

In [ ]:
from tabular_agent.io import load_table
from tabular_agent.harness.tools import _feature_frame

for name, target, path in [
    ("rossmann",   "Sales",    "_runs/capacity_rossmann/tables/t1.parquet"),
    ("restaurant", "visitors", "_runs/capacity_restaurant/data/work.parquet"),
]:
    df = load_table(os.path.join(REPO, path))
    X, dropped = _feature_frame(df, target)
    print(f"{name:<11} {X.shape[1]:>3} numeric features")
    print(f"            kept   : {list(X.columns)[:6]} ...")
    print(f"            dropped: {dropped[:4]} ...")
    print()

There it is. **`Store` is an integer, so it survives. `air_store_id` is a string, so it
is dropped.**

Rossmann's baseline carries 1115 store identities. Restaurant's baseline carries **no
entity identity at all**. The two datasets differ on *two* axes at once — rows and
identity — which is exactly the confound 0017 forbids, and either one could explain the
result.

So: give restaurant the same *kind* of column and nothing else.

Not `fit_encoding` — a target encoding adds information as well as identity, and would
put the confound straight back. What is wanted is what `Store` already is: an arbitrary
integer label carrying no meaning beyond "these rows are the same shop".

### The harness refused my first attempt, and was right

The obvious way to write that is `factorize`:

```python
df['air_store_code'] = df['air_store_id'].factorize(sort=True)[0]
```

```
SandboxError: these columns depend on which other rows are in the frame:
['air_store_code']. The same rows produced different values when transformed as
part of a smaller batch, so at predict time — where the batch is whatever
arrives, sometimes a single row — they would not match what the model trained on.
```

0011's stability check. A **positional** code depends on which rows are present: the
same shop gets a different number in a batch of one. The refusal is correct, and it is
worth more than the convenience.

The fix is to derive the code from the identifier itself — a pure function of the row:

In [ ]:
ids = pd.read_parquet(os.path.join(REPO, "restaurant_visitor", "train_joined.parquet"),
                      columns=["air_store_id"])["air_store_id"]
code = ids.str[4:12].apply(int, base=16)
print(f"{ids.nunique()} distinct ids -> {code.nunique()} distinct codes (no collisions)")
print(ids.head(3).tolist(), "->", code.head(3).tolist())

## 5. It is not rows. It is per-entity effects.

In [ ]:
print(table_for("restaurant", "baseline table"))
print()
print(table_for("restaurant", "+ store code"))

| restaurant | 200 trees | best budget | gain |
|---|---|---|---|
| baseline | 0.76728 | 0.76693 | **+0.0004** |
| + store code | 0.62897 | 0.60064 | **+0.0283** |

The same rows. The same split. The same validation block. One arbitrary integer column,
and capacity goes from worth nothing to worth 0.028 rmsle.

> **200 trees cannot reach 829 shop effects, let alone Rossmann's 1115.** Capacity pays
> where there is high-cardinality *identity* to learn — a property of the table, not of
> its size.

Which explains the rest of the grid. house-prices (1,168 houses, no repeated entity) and
Porto Seguro (595k anonymised drivers, one row each) have no such structure and gain
nothing. Every forecasting table we build has it.

A caution on reading that number against the competition, because it is tempting and
wrong. The leaderboard is 2,150 teams packed between 0.46535 and a median of 0.4892 — a
top two-thirds spanning 0.023 rmsle, so in *that* region 0.028 would be worth most of the
field. Our 0.60 is nowhere near that region: it is a raw-numeric baseline scored on our
own validation block, not a submission scored on the competition's test set, and it sits
in the sparse tail where the same 0.028 moves only ~30 places.


In [ ]:
from prep_restaurant import leaderboard_rank
for s in [0.62897, 0.60064]:
    print(f"{s:.5f} -> {leaderboard_rank(s)}")

## 6. The finding nobody was looking for

The plan wanted capacity swept on `t24`, v8's best Rossmann table. That turned out to be
impossible — see `0055`: 0051's column rename landed four hours after v8's tables were
written, so their stored lineages emit `_latest` while the tables hold `_last`, and
0030's servability check refuses them. Correctly.

So the second reading is a **rebuild** of the one part of that pipeline that is pure code
and therefore version-independent: the agent's own feature block, including its
`Store × DayOfWeek` and `Store × Promo` interactions. Same source, same fractions, same
`split_on` — so a bit-identical split.

In [ ]:
for d in ["capacity_rossmann", "capacity_rossmann_rich"]:
    v = pd.read_parquet(os.path.join(REPO, "_runs", d, "data", "val.parquet"))
    print(f"{d:<24} val {len(v):,} rows, {v.Date.min().date()} -> {v.Date.max().date()}")

In [ ]:
base = results[(results.dataset == "rossmann") & (results.table == "baseline table")]
rich = results[(results.dataset == "rossmann") & (results.table == "v8 feature block")]

cmp = pd.DataFrame({
    "budget": base.budget.str.strip().values[:5],
    "baseline": base.score.values[:5],
    "+ feature block": rich.score.values[:5],
})
cmp["features_worth"] = cmp["baseline"] - cmp["+ feature block"]
print(cmp.to_string(index=False))

Read the last column. It is what the agent's feature engineering was worth, at each
budget, in rmspe — positive means the features helped.

| Rossmann, rmspe | baseline | + feature block | features worth |
|---|---|---|---|
| **200 trees (the harness today)** | 0.21810 | 0.23331 | **−0.0152 — worse** |
| 5000 @ lr 0.03 | 0.16083 | 0.15213 | **+0.0087 — better** |

**The sign reverses.**

At the budget this harness actually uses, the agent's feature work scores as *harmful*.
At a real budget, the same columns on the same rows against the same validation block are
worth having.

This is 0022's failure one level up, and worse than 0053's. 0053 found a feature's
measured worth *moving* with the fold count. This is a feature's measured worth
**changing sign** with a constant nobody chose — in the ruler that ranks every table in
every run, and whose argmax (`best_table`) decides what gets bundled and shipped.

Note too that capacity is worth *more* on the engineered table (+0.0812) than on the bare
one (+0.0573). Features and capacity were never competing explanations for our mediocre
scores. The features were being **suppressed by the budget**.

## 7. Why "just raise the constant" is not the fix

Collect what early stopping asked for, across all five readings.

In [ ]:
es = results[results.budget.str.contains("early stop")].copy()
es["rounds"] = es.budget.str.extract(r"(\d+)").astype(int)
print(es[["dataset", "table", "rounds", "score", "gain_vs_current"]].to_string(index=False))

**162, 181, 1018, 3430, 16926.** Two orders of magnitude.

Any constant large enough for Rossmann would wreck house-prices and Porto Seguro, where
*every* raised budget loses. Any constant small enough for those two leaves 0.06 rmspe on
the table on Rossmann. There is no defensible constant — and picking the one that suits
Rossmann would be a leaderboard-fit, which is the failure CLAUDE.md and 0022 exist to
prevent, one level up.

**Early stopping wins or ties on all five readings and is never worse than the
incumbent**, including on the datasets where capacity is actively harmful — because it
declines to spend it. It is the only arm here with that property, and it needs no
per-dataset knowledge.

That is 0039's question, answered: not a common budget, a *chosen* one.

## 8. What is deliberately not decided here

**How the harness gets an early-stopping block.** Early stopping needs validation rows
the fit is allowed to see, and this harness has strong opinions about which rows a fit
may look at (0023, 0029, 0041, 0043). The arm measured here carves the tail 10% off the
work half, which is defensible for an ordered split and needs an argument for an
unordered one. That is a design decision with consequences for the sealed holdout and for
what a bundle stores — and 0035 removed model parameters from the agent on purpose, so
this is *not* an argument for handing them back.

**What this does to sixteen runs of results.** Every table comparison in `_runs/` was made
at 200 trees. The Rossmann sign reversal says at least some of those rankings are not
merely noisy but inverted. 0052 and 0053's conclusions are about features measured by
this ruler and inherit the problem directly.

Datasets are probes, not targets: the output of this notebook is not a setting to adopt.
It is the finding that a constant nobody chose has been deciding what our instrument
says.